# Recherche automatique de sources (Xanto) — exécution sur Colab

Méthode et règles : `PREINSCRIPTION.md` et `PREINSCRIPTION_ETAPE2.md`. Les images sont téléchargées à leur source publique (musées, catalogue de Rovigo) d'après `manifeste_images.csv`, empreintes vérifiées ; rien n'est republié. Environnement : Exécution > Modifier le type d'exécution > GPU T4, puis Exécution > Tout exécuter.

In [ ]:
!nvidia-smi -L || echo 'PAS DE GPU'

In [ ]:
!rm -rf /content/xanto && git clone -q https://github.com/maribakulj/xanto /content/xanto && cd /content/xanto && git log -1 --format=%h
# estampes du Met (CC0) : l'API du Met refuse les serveurs de Colab, on prend la copie publiée en « release » du dépôt
!mkdir -p /content/xanto-rs/recherche-sources && cd /content/xanto-rs/recherche-sources && wget -q https://github.com/maribakulj/xanto/releases/download/galerie-met-2026-10-13/met.zip && unzip -q -o met.zip && rm met.zip && ls met | wc -l
!python /content/xanto/recherche-sources/telecharger_donnees.py /content/xanto-rs

In [ ]:
!ls /content/xanto-rs/recherche-sources | head; ls /content/xanto-rs/recherche-sources/met | wc -l; ls /content/xanto-rs/corpus-images/rovigo | wc -l

In [ ]:
# Étape 1 (méthode préinscrite, DINOv2-base)
!cd /content && python xanto/recherche-sources/recherche.py /content/xanto-rs/recherche-sources 2>&1 | grep -v Warning | tail -8

In [ ]:
# Étape 2 (reclassement par détails cohérents)
!cd /content && python xanto/recherche-sources/etape2.py /content/xanto-rs/recherche-sources 2>&1 | grep -v Warning | tail -6

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, glob
OUT_LOCAL = '/content/drive/MyDrive/xanto-resultats'
os.makedirs(OUT_LOCAL, exist_ok=True)
for f in glob.glob('/content/xanto-rs/recherche-sources/resultats_*.csv'): shutil.copy(f, OUT_LOCAL)
print(os.listdir(OUT_LOCAL))

## Exploratoire : DINOv3 (ne compte pas pour la décision)
Nécessite le secret Colab `HF_TOKEN` (icône clé à gauche) d'un compte ayant accepté la licence de `facebook/dinov3-vitb16-pretrain-lvd1689m`.

In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(userdata.get('HF_TOKEN'))
!cd /content && python xanto/recherche-sources/recherche.py /content/xanto-rs/recherche-sources facebook/dinov3-vitb16-pretrain-lvd1689m 2>&1 | grep -v Warning | tail -6
!cp /content/xanto-rs/recherche-sources/resultats_*.csv /content/drive/MyDrive/xanto-resultats/